# Drift Detection — 02: Bayesian Online Changepoint Detection (BOCPD)

Instead of hand-coded signal patterns, BOCPD models the **posterior probability that a changepoint
occurred at each time step**. It maintains a "run length" distribution — how long the current
regime has been active — and updates it with each new observation.

The signal taxonomy (crash / fade / no-recovery) is **not an input**. These patterns emerge
from the posterior shape — a crash produces a sudden spike in P(changepoint); a fade produces
a gradual rise over multiple steps.

**Doc reference:** `docs/evolution/drift_detection.md § 3.2`

### Implementation

Uses a **Dirichlet-Categorical conjugate prior** for {-1, 0, +1} alignment scores.
- Prior: Dirichlet(α₀) — uninformative (α₀ = [1, 1, 1])
- Likelihood: Categorical(θ) — probability of each score value in the current regime
- Hazard rate: constant H (probability of changepoint at each step)
- At each step: update run-length distribution using Bayes' theorem

### Data note

With ~10-15 steps per persona per dimension, posterior estimates will be wide.
BOCPD is most powerful at ~50+ personas with confirmed changepoints. This notebook
validates the implementation and shows the qualitative behaviour on current data.


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.9)

# --- Load judge labels (204 personas, 1651 entries, integer {-1, 0, 1} scores) ---
# Using judge labels rather than human annotations: more data (all 204 personas vs 24),
# and clean integer scores (no averaging artefacts).

VALUE_COLS = [
    "alignment_self_direction", "alignment_stimulation", "alignment_hedonism",
    "alignment_achievement", "alignment_power", "alignment_security",
    "alignment_conformity", "alignment_tradition", "alignment_benevolence",
    "alignment_universalism",
]
SHORT_NAMES = [c.replace("alignment_", "") for c in VALUE_COLS]
DIM_LABELS  = ["SD", "ST", "HE", "AC", "PO", "SE", "CO", "TR", "BE", "UN"]

judge_df = pl.read_parquet("../../logs/judge_labels/judge_labels.parquet")
mean_df  = (
    judge_df
    .select(["persona_id", "t_index"] + VALUE_COLS)
    .with_columns([pl.col(c).cast(pl.Float64) for c in VALUE_COLS])
    .sort(["persona_id", "t_index"])
)

registry   = pl.read_parquet("../../logs/registry/personas.parquet").select(
                ["persona_id", "name", "core_values"])
id_to_name = dict(zip(registry["persona_id"].to_list(), registry["name"].to_list()))
id_to_core = dict(zip(registry["persona_id"].to_list(), registry["core_values"].to_list()))

persona_ids = sorted(mean_df["persona_id"].unique().to_list())
print(f"Loaded {len(judge_df)} judge-labelled entries across {len(persona_ids)} personas")

# --- Sample: verify input scores look correct ---
# Pick first persona with ≥5 entries so the sample is representative
sample_pid = next(
    pid for pid in persona_ids
    if mean_df.filter(pl.col("persona_id") == pid).height >= 5
)
sample_data = mean_df.filter(pl.col("persona_id") == sample_pid).head(5)
print(f"\nSample scores (first 5 entries) for '{id_to_name.get(sample_pid, sample_pid)}':")
print(f"  Core values: {id_to_core.get(sample_pid, [])}")
print(sample_data.select(["t_index"] + VALUE_COLS))
print("  Scores are integers in {{-1, 0, +1}} across all 10 Schwartz dimensions")


def get_persona_matrix(pid: str) -> tuple[list[int], np.ndarray]:
    pdata = mean_df.filter(pl.col("persona_id") == pid).sort("t_index")
    return pdata["t_index"].to_list(), np.array([pdata[c].to_list() for c in VALUE_COLS]).T


def get_profile_weights(pid: str) -> np.ndarray:
    core = id_to_core.get(pid, [])
    name_to_idx = {s.lower().replace("-", "_"): i for i, s in enumerate(SHORT_NAMES)}
    w = np.zeros(10)
    for v in core:
        key = v.lower().replace("-", "_").replace(" ", "_")
        if key in name_to_idx:
            w[name_to_idx[key]] = 1.0
    if w.sum() > 0:
        w /= w.sum()
    return w


def core_dim_indices(weights: np.ndarray, w_min: float = 0.15) -> list[int]:
    return [j for j, wj in enumerate(weights) if wj >= w_min]


## BOCPD Implementation

Dirichlet-Categorical conjugate update for ordinal {-1, 0, +1} scores.
The run-length posterior `R[t]` is a distribution over how many steps the current regime has lasted.

In [ ]:
def bocpd_dimension(
    scores_1d: np.ndarray,
    hazard: float = 0.1,
    alpha0: np.ndarray | None = None,
) -> dict:
    """
    Bayesian Online Changepoint Detection with Dirichlet-Categorical likelihood.

    Correct formulation (Fearnhead & Liu, 2007):
      P(r_t=0, x_{1:t}) = H * p(x_t | alpha0) * sum_r P(r_{t-1}=r, x_{1:t-1})

    The new-run predictive uses alpha0 (fresh regime prior), NOT the old run's
    posterior — that was the bug in the earlier version.

    Parameters
    ----------
    scores_1d : (T,) array of alignment scores — integers {-1, 0, +1}
    hazard    : prior probability of changepoint at each step (H)
    alpha0    : Dirichlet prior pseudo-counts for bins [-1, 0, +1]. Default [1,1,1].

    Returns
    -------
    dict with key:
        p_change : (T,) — P(changepoint at t | data_{1:t})
    """
    if alpha0 is None:
        alpha0 = np.array([1.0, 1.0, 1.0])

    T = len(scores_1d)

    def score_to_bin(s: float) -> int:
        if s < -0.5: return 0
        if s <  0.5: return 1
        return 2

    # Each particle: (alpha posterior for this run)
    # log_weights[i] = log P(run i, x_{1:t-1}) — normalised after each step
    alphas      = [alpha0.copy()]
    log_weights = np.array([0.0])   # starts normalised (single run, weight 1)

    p_change = np.zeros(T)

    log_p_new = np.log(alpha0[0] / alpha0.sum())  # placeholder, recomputed each step

    for t in range(T):
        obs_bin = score_to_bin(scores_1d[t])

        # Predictive log-prob under each existing run
        log_preds = np.array([np.log(a[obs_bin] / a.sum()) for a in alphas])

        # --- Changepoint term ---
        # P(r_t=0, x_{1:t}) = H * p(x_t | alpha0) * sum_r exp(log_weights[r])
        # Since weights are normalised, sum = 1 → log_sum = 0
        log_new = np.log(hazard) + np.log(alpha0[obs_bin] / alpha0.sum())

        # --- Continue terms ---
        log_continue = log_weights + log_preds + np.log(1 - hazard)

        # --- Combine and normalise ---
        all_lw = np.append(log_continue, log_new)
        max_lw = all_lw.max()
        w_unnorm = np.exp(all_lw - max_lw)
        w_norm   = w_unnorm / w_unnorm.sum()

        p_change[t] = w_norm[-1]   # weight of the new run = P(changepoint at t)

        # --- Update alphas and weights ---
        new_alphas = []
        for i, a in enumerate(alphas):
            upd = a.copy(); upd[obs_bin] += 1
            new_alphas.append(upd)
        # New run has already observed x_t
        new_run_alpha = alpha0.copy(); new_run_alpha[obs_bin] += 1
        new_alphas.append(new_run_alpha)

        alphas      = new_alphas
        log_weights = np.log(np.maximum(w_norm, 1e-300))

    return {"p_change": p_change}


## Why BOCPD Has No Training Phase

BOCPD is a **fully online** algorithm: it updates its beliefs with each new observation and
requires no separate training step. This is a fundamental design difference from supervised
approaches (like the Critic) or batch methods (like the Autoencoder).

What BOCPD does instead of training:

| Rule-based equivalent | BOCPD equivalent |
|---|---|
| Grid search over δ, τ_low, C_min | Grid search over **H** (hazard rate) and **α₀** (Dirichlet prior) |
| Consensus hit-rate tuning | Consensus hit-rate tuning — same metric, different parameters |
| Threshold picked once, applied online | H picked once, applied online |

**H (hazard rate):** prior probability of a changepoint at each step.
- Low H (e.g. 0.05) → conservative: needs strong evidence before flagging.
- High H (e.g. 0.40) → sensitive: fires frequently, higher false-positive rate.

**α₀ (Dirichlet prior):** initial belief about score distribution within a regime.
- Uninformative (α₀ = [1, 1, 1]) → equal prior over {-1, 0, +1}.
- Informative (e.g. α₀ = [1, 2, 4]) → prior belief that aligned scores are more likely.

The cell below selects H by grid search against consensus labels derived from
the 6 rule-based sub-approaches (same method as notebook 01).


In [ ]:
# --- Inline consensus from rule-based approaches (mirrors notebook 01) ---
# We need a ground-truth proxy to tune H. Recompute it here using the same 6 sub-approaches.

def _ema_alerts(matrix, w, alpha=0.3, threshold=0.10, w_min=0.15):
    T, K = matrix.shape
    ema = np.zeros(K)
    alerts = set()
    for t in range(T):
        for j in range(K):
            if w[j] < w_min: continue
            ema[j] = alpha * w[j] * max(0., -matrix[t, j]) + (1 - alpha) * ema[j]
            if ema[j] > threshold: alerts.add(t)
    return alerts

def _cusum_alerts(matrix, w, k=0.3, h=1.5, w_min=0.15):
    T, K = matrix.shape
    jar = np.zeros(K)
    alerts = set()
    for t in range(T):
        for j in range(K):
            if w[j] < w_min: continue
            jar[j] = max(0., jar[j] + w[j] * max(0., -matrix[t, j]) - k)
            if jar[j] > h: alerts.add(t)
    return alerts

def _cosine_alerts(matrix, w, threshold=0.0):
    alerts = set()
    for t in range(len(matrix)):
        a = matrix[t]; norm = np.linalg.norm(w) * np.linalg.norm(a)
        if norm > 1e-8 and np.dot(w, a) / norm < threshold: alerts.add(t)
    return alerts

def _cc_alerts(matrix, w, baseline_end=3, n_sigma=2.0, w_min=0.15):
    T, K = matrix.shape
    if baseline_end >= T: return set()
    mu = matrix[:baseline_end].mean(0); sig = matrix[:baseline_end].std(0)
    alerts = set()
    for t in range(baseline_end, T):
        for j in range(K):
            if w[j] >= w_min and matrix[t, j] < mu[j] - n_sigma * sig[j]:
                alerts.add(t)
    return alerts

def _kl_alerts(matrix, w, baseline_end=3, window=3, kl_thresh=0.15, w_min=0.15):
    from scipy.special import rel_entr
    T, K = matrix.shape
    bins = [-1.5, -0.5, 0.5, 1.5]
    def dist(vals):
        c, _ = np.histogram(vals, bins=bins); d = c.astype(float) + 0.05
        return d / d.sum()
    alerts = set()
    for j in range(K):
        if w[j] < w_min: continue
        bd = dist(matrix[:baseline_end, j])
        for t in range(baseline_end + window, T + 1):
            if np.sum(rel_entr(dist(matrix[t-window:t, j]), bd)) > kl_thresh:
                alerts.add(t - 1)
    return alerts

def _baseline_alerts(matrix, w, delta=0.5, tau=-0.4, c_min=3, w_min=0.15):
    T, K = matrix.shape
    scalar = matrix @ w; state = np.zeros(K); alerts = set()
    for t in range(1, T):
        if scalar[t-1] - scalar[t] > delta: alerts.add(t)
        for j in range(K):
            if w[j] < w_min: state[j] = 0; continue
            if matrix[t, j] < tau: state[j] += 1;
            else: state[j] = 0
            if state[j] >= c_min: alerts.add(t)
    return alerts

def compute_consensus(pid):
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < 5: return {}
    w = get_profile_weights(pid)
    be = max(2, min(4, T // 3))
    votes = {t: 0 for t in range(T)}
    for fn, kw in [
        (_baseline_alerts, {}), (_ema_alerts, {}), (_cusum_alerts, {}),
        (_cosine_alerts, {}), (_cc_alerts, {"baseline_end": be}),
        (_kl_alerts, {"baseline_end": be}),
    ]:
        for t in fn(matrix, w, **kw):
            if t in votes: votes[t] += 1
    return votes


# --- Grid search over H and cp_thresh ---
print("Grid searching H × cp_thresh against consensus labels...\n")

hazard_grid  = [0.05, 0.10, 0.15, 0.25, 0.40]
thresh_grid  = [0.30, 0.40, 0.50, 0.60]
w_min_gs     = 0.15

# Cache BOCPD outputs per (pid, j, H) to avoid recomputation
from functools import lru_cache

bocpd_cache = {}
for H in hazard_grid:
    for pid in persona_ids:
        t_idx, matrix = get_persona_matrix(pid)
        if len(t_idx) < 5: continue
        w = get_profile_weights(pid)
        for j in core_dim_indices(w, w_min_gs):
            bocpd_cache[(pid, j, H)] = bocpd_dimension(matrix[:, j], hazard=H)["p_change"]

grid_results = []
for H in hazard_grid:
    for cp_thresh in thresh_grid:
        tp = fp = fn = tn = 0
        for pid in persona_ids:
            t_idx, matrix = get_persona_matrix(pid)
            T = len(t_idx)
            if T < 5: continue
            w  = get_profile_weights(pid)
            sv = compute_consensus(pid)
            crisis     = {t for t, s in sv.items() if s >= 4}
            non_crisis = {t for t, s in sv.items() if s < 4}

            alerted = set()
            for j in core_dim_indices(w, w_min_gs):
                p_change = bocpd_cache.get((pid, j, H), [])
                for t, pc in enumerate(p_change):
                    if pc > cp_thresh: alerted.add(t)

            tp += len(alerted & crisis); fp += len(alerted & non_crisis)
            fn += len(crisis - alerted); tn += len(non_crisis - alerted)

        hit  = tp / max(tp + fn, 1)
        prec = tp / max(tp + fp, 1)
        f1   = 2 * hit * prec / max(hit + prec, 1e-9)
        fpr  = fp / max(fp + tn, 1)
        grid_results.append({"H": H, "cp_thresh": cp_thresh,
                              "hit": hit, "prec": prec, "f1": f1, "fpr": fpr})

print(f"{'H':>5s}  {'Thresh':>6s}  {'Hit%':>5s}  {'Prec%':>5s}  {'F1':>6s}  {'FPR%':>5s}")
print("-" * 50)
best = max(grid_results, key=lambda r: r["f1"])
for r in grid_results:
    marker = " ←" if r == best else ""
    print(f"{r['H']:>5.2f}  {r['cp_thresh']:>6.2f}  {r['hit']*100:>4.1f}%  "
          f"{r['prec']*100:>4.1f}%  {r['f1']:>6.3f}  {r['fpr']*100:>4.1f}%{marker}")

HAZARD    = best["H"]
CP_THRESH = best["cp_thresh"]
print(f"\nSelected H={HAZARD}, cp_thresh={CP_THRESH} (best F1={best['f1']:.3f})")


## Run BOCPD on All Personas

Using the selected hazard rate H from grid search above.
For each persona × core dimension, compute P(changepoint at t) across all time steps.

In [ ]:
# HAZARD and CP_THRESH are set by grid search above
W_MIN = 0.15

bocpd_results = {}

for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < 5:
        continue

    w = get_profile_weights(pid)
    core_j = core_dim_indices(w, W_MIN)

    dim_results = {}
    alerts = []

    for j in core_j:
        out = bocpd_dimension(matrix[:, j], hazard=HAZARD)
        dim_results[j] = out
        for t, pc in enumerate(out["p_change"]):
            if pc > CP_THRESH:
                alerts.append((t, j))

    bocpd_results[pid] = {
        "name":       id_to_name.get(pid, pid[:8]),
        "core":       id_to_core.get(pid, []),
        "T":          T,
        "t_idx":      t_idx,
        "matrix":     matrix,
        "weights":    w,
        "dim_results": dim_results,
        "alerts":     alerts,
    }

print(f"BOCPD ran on {len(bocpd_results)} personas\n")
print(f"{'Persona':<25s}  {'Core':<28s}  T  Alerts")
print("-" * 70)
for pid, r in bocpd_results.items():
    print(f"{r['name']:<25s}  {', '.join(r['core']):<28s}  {r['T']:2d}  {len(r['alerts'])}")


## Visualise P(changepoint) Over Time

For each persona, show alignment scores on core dimensions alongside P(changepoint).
Red shading = P(changepoint) > threshold.

In [ ]:
palette = sns.color_palette("tab10", n_colors=10)


def plot_bocpd_persona(pid: str):
    r = bocpd_results[pid]
    matrix = r["matrix"]
    T = r["T"]
    core_j = [j for j in r["dim_results"]]
    if not core_j:
        return

    fig, axes = plt.subplots(len(core_j), 2, figsize=(14, 2.5 * len(core_j)),
                             sharex=True, gridspec_kw={"width_ratios": [2, 1]})
    if len(core_j) == 1:
        axes = [axes]

    for ax_row, j in zip(axes, core_j):
        ax_score, ax_cp = ax_row
        p_change = r["dim_results"][j]["p_change"]

        # Score plot
        ax_score.plot(range(T), matrix[:, j], "o-", color=palette[j], lw=1.8)
        ax_score.axhline(0, color="grey", lw=0.5, ls="--")
        ax_score.set_ylim(-1.4, 1.4)
        ax_score.set_ylabel(DIM_LABELS[j], fontsize=9)

        # Shade high P(changepoint) regions
        for t, pc in enumerate(p_change):
            if pc > CP_THRESH:
                ax_score.axvspan(t - 0.4, t + 0.4, color="red", alpha=pc * 0.4, zorder=1)

        # P(changepoint) plot
        ax_cp.bar(range(T), p_change, color="salmon", alpha=0.8)
        ax_cp.axhline(CP_THRESH, color="red", lw=1, ls="--")
        ax_cp.set_ylim(0, 1)
        ax_cp.set_ylabel("P(change)", fontsize=8)
        ax_cp.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))

    axes[0][0].set_title(f"{r['name']} — {', '.join(r['core'])}", fontweight="bold")
    axes[-1][0].set_xlabel("t_index")
    axes[-1][1].set_xlabel("t_index")
    plt.tight_layout()
    plt.show()


for pid in list(bocpd_results.keys())[:4]:
    plot_bocpd_persona(pid)


## Sensitivity Analysis: Hazard Rate

The hazard rate H controls how often the model expects changepoints a priori.
Low H = conservative (needs strong evidence). High H = sensitive (fires more often).

In [ ]:
hazard_values = [0.05, 0.10, 0.15, 0.25, 0.40]

# Pick first persona with ≥6 steps and ≥1 core dimension
example_pid = next(
    pid for pid, r in bocpd_results.items()
    if r["T"] >= 6 and r["dim_results"]
)
example_r = bocpd_results[example_pid]
example_j = list(example_r["dim_results"].keys())[0]
scores_1d  = example_r["matrix"][:, example_j]
T          = example_r["T"]

fig, axes = plt.subplots(len(hazard_values), 1,
                         figsize=(12, 2.2 * len(hazard_values)), sharex=True)

for ax, H in zip(axes, hazard_values):
    out = bocpd_dimension(scores_1d, hazard=H)
    ax.bar(range(T), out["p_change"], color="salmon", alpha=0.8, label=f"H={H}")
    ax.axhline(CP_THRESH, color="red", lw=1, ls="--")
    ax.set_ylim(0, 1)
    ax.set_ylabel(f"H={H}", fontsize=9)
    ax2 = ax.twinx()
    ax2.plot(range(T), scores_1d, "o-", color="steelblue", ms=4, lw=1.2)
    ax2.set_ylim(-1.4, 1.4)
    ax2.set_ylabel("score", fontsize=8)

axes[0].set_title(
    f"{example_r['name']} — {DIM_LABELS[example_j]}: P(changepoint) vs hazard rate",
    fontweight="bold")
axes[-1].set_xlabel("t_index")
plt.tight_layout()
plt.show()


## Limitations

- **Small data:** ~10-15 steps per persona per dimension. Posterior is prior-dominated early on.
- **Discrete scores:** {-1, 0, +1} means regime estimates are noisy on short windows.
- **No profile weights `w_u` natively:** BOCPD operates per-dimension independently. Profile weighting requires a wrapper (e.g., suppress alerts on low-weight dimensions).
- **Interpretability:** `P(changepoint) = 0.87` needs translation to user-facing language for the Coach.

At ~50+ personas with confirmed changepoints, BOCPD becomes the recommended upgrade from rule-based (see `docs/evolution/drift_detection.md § 3.2`).